<a href="https://colab.research.google.com/github/springboardmentor12458j/LiveMeetingSummarize/blob/varshini/Module_2_Milestone_1_ai_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit streamlit-webrtc av pydub openai-whisper python-dotenv -q
!pip install pyngrok -q
!pip install audio-recorder-streamlit


In [169]:
from google.colab import userdata
import os

# Get ngrok auth token from Colab Secrets (secure method)
ngrok_auth_token = userdata.get('NGROK_AUTH_TOKEN')

if not ngrok_auth_token:
    print("⚠️ ngrok auth token not found!")
    print("Please add it to Colab Secrets:")
    print("1. Click '🔑' icon on left panel")
    print("2. Add new secret named 'NGROK_AUTH_TOKEN'")
    print("3. Paste your token from https://dashboard.ngrok.com/auth")
    raise ValueError("NGROK_AUTH_TOKEN not configured")

# Set ngrok auth
os.environ['NGROK_AUTHTOKEN'] = ngrok_auth_token
print("✅ ngrok configured securely from Colab Secrets")


✅ ngrok configured securely from Colab Secrets


In [170]:

import streamlit as st
from stt_engine import STTEngine
from audio_recorder_streamlit import audio_recorder
import tempfile
import os
from datetime import datetime

st.set_page_config(page_title="Live Speech-to-Text", page_icon="🎙️", layout="wide")
st.title("🎙️ Live Speech-to-Text")
st.info("🎤 Click to Record → Speak → Get Transcript Instantly")

# Initialize STT Engine
if "stt_engine" not in st.session_state:
    try:
        with st.spinner("Loading Whisper Model (first time: 30-60 seconds)..."):
            st.session_state["stt_engine"] = STTEngine(model_size="base")
        st.success("✅ Whisper STT Engine Loaded!")
    except Exception as e:
        st.error(f"Failed to load STT Engine: {e}")
        st.stop()

st.markdown("---")

# Audio Recorder - Direct Recording Only
st.markdown("### 🎙️ Record Your Audio")
audio_bytes = audio_recorder(
    text="Click to record",
    recording_color="#e74c3c",
    neutral_color="#3498db",
    icon_name="microphone",
    icon_size="3x"
)

# Direct transcription when audio is recorded
if audio_bytes:
    try:
        with st.spinner("🎵 Transcribing audio..."):
            # Save audio bytes to temporary file
            with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as tmp:
                tmp.write(audio_bytes)
                temp_path = tmp.name

            # Transcribe the audio directly
            transcript = st.session_state["stt_engine"].transcribe_file(temp_path)

            # Clean up temp file
            try:
                os.unlink(temp_path)
            except:
                pass

        if not transcript or "No speech detected" in transcript:
            st.warning("⚠️ No speech detected! Please try recording again.")
        else:
            st.success("✅ Transcription Complete!")
            st.markdown("---")
            st.markdown("### 📝 Transcript")
            st.markdown(f"**{transcript}**")

            # Download transcript text only
            st.download_button(
                label="📥 Download Transcript",
                data=transcript,
                file_name=f"transcript_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt",
                mime="text/plain",
                use_container_width=True
            )

    except Exception as e:
        st.error(f"❌ Error during transcription: {str(e)}")

st.markdown("---")
st.markdown("### 📖 How to Use")
st.markdown("1️⃣ Click the **microphone button**")
st.markdown("2️⃣ **Speak clearly**")
st.markdown("3️⃣ Click again to **stop**")
st.markdown("4️⃣ **View transcript instantly!**")

2026-01-01 08:59:48.864 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-01 08:59:48.865 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-01 08:59:48.867 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-01 08:59:48.868 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-01 08:59:48.870 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-01 08:59:48.872 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-01 08:59:48.873 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-01 08:59:48.877 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()

In [171]:
import os

# Create .streamlit directory
os.makedirs('.streamlit', exist_ok=True)

config_content = '''
[client]
showErrorDetails = true
allowRunOnSave = true

[server]
headless = true
port = 8501
enableXsrfProtection = false

[logger]
level = "info"
'''

with open('.streamlit/config.toml', 'w') as f:
    f.write(config_content)

print("✅ Streamlit config created")


✅ Streamlit config created


In [172]:
# Add this as a new cell
import subprocess
print("System audio devices:")
subprocess.run(['python', '-m', 'sounddevice'], capture_output=True)


System audio devices:


CompletedProcess(args=['python', '-m', 'sounddevice'], returncode=1, stdout=b'', stderr=b'/usr/bin/python3: No module named sounddevice\n')

In [173]:
# Debug: Check if microphone is accessible
import subprocess
result = subprocess.run(['python', '-c', '''
import sounddevice as sd
import numpy as np

print("Testing microphone...")
try:
    duration = 2
    fs = 44100
    recording = sd.rec(int(duration * fs), samplerate=fs, channels=1)
    sd.wait()
    print(f"✅ Microphone works! Captured {len(recording)} samples")
except Exception as e:
    print(f"❌ Microphone error: {e}")
'''], capture_output=True, text=True)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)



STDERR: Traceback (most recent call last):
  File "<string>", line 2, in <module>
ModuleNotFoundError: No module named 'sounddevice'



In [174]:
import subprocess
import time
from pyngrok import ngrok

# Kill any existing streamlit processes
!pkill -f streamlit

# Kill any existing ngrok tunnels to prevent conflicts
ngrok.kill()

# Start Streamlit in background
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port=8501'])

# Give streamlit time to start
time.sleep(3)

# Create private ngrok tunnel
print("🔗 Creating private ngrok tunnel...")
public_url = ngrok.connect(8501, "http")
print(f"\n{'='*60}")
print(f"✅ STREAMLIT APP IS LIVE!")
print(f"{'='*60}")
print(f"\n🔐 Private Tunnel URL: {public_url}")
print(f"\nClick here to open: {public_url}")
print(f"\n{'='*60}")
print(f"⚠️  This URL is PRIVATE - only you can access it")
print(f"{'='*60}\n")

🔗 Creating private ngrok tunnel...

✅ STREAMLIT APP IS LIVE!

🔐 Private Tunnel URL: NgrokTunnel: "https://mistilled-unimported-milagros.ngrok-free.dev" -> "http://localhost:8501"

Click here to open: NgrokTunnel: "https://mistilled-unimported-milagros.ngrok-free.dev" -> "http://localhost:8501"

⚠️  This URL is PRIVATE - only you can access it

